# Day 17 — Memory Management & Accumulators

**What you will learn:**
- How Spark divides executor memory: storage, execution, user
- Unified Memory Model (Spark 1.6+): dynamic allocation between storage and execution
- Accumulators: shared counters/sums from tasks back to the driver
- Custom accumulators
- Common memory tuning knobs

**Exam relevance:** Architecture (20%) + Optimisation (10%) — memory model fractions, accumulator semantics, and GC tuning are tested.

## 1. Executor Memory Layout

```
spark.executor.memory  (e.g. 4g)
│
├── Reserved Memory     — 300 MB fixed (JVM overhead)
│
└── Usable Memory  (executor.memory - 300MB)
    │
    ├── spark.memory.fraction (default 0.6)
    │   Unified Memory — shared between:
    │   ├── Execution Memory  — shuffle buffers, sort, join
    │   └── Storage Memory    — cache, broadcast variables
    │   (they borrow from each other dynamically)
    │
    └── User Memory (1 - 0.6 = 0.4)
        UDFs, data structures in application code
```

> **Exam tip:** `spark.memory.fraction` (default 0.6) controls the unified pool. `spark.memory.storageFraction` (default 0.5) sets the initial storage/execution split *within* the unified pool — but they can borrow from each other.

In [ ]:
# Inspect current memory configuration
configs = [
    "spark.executor.memory",
    "spark.driver.memory",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
    "spark.executor.memoryOverhead",
    "spark.sql.shuffle.partitions",
]
for k in configs:
    try:
        print(f"{k}: {spark.conf.get(k)}")
    except Exception:
        print(f"{k}: (not set — using default)")

## 2. GC Pressure

- High GC time (visible in Spark UI Executors tab) means the JVM is spending too much time reclaiming objects
- Primary cause: too many short-lived objects in user code or UDFs
- Mitigation: use built-in functions (stay in JVM), reduce user memory fraction, or use Kryo serializer

```python
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
```

## 3. Accumulators — Aggregating Values from Tasks

Accumulators are write-only from tasks, read-only from the driver.

**Use cases:**
- Counting bad records without collecting data to driver
- Tracking events during a large transformation
- Debug counters in production jobs

> **Exam tip:** Accumulators in transformations (lazy) can be updated **multiple times** if Spark re-executes a task. Use accumulators only in **actions** for reliable counts.

In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

# Long-type accumulator
null_count    = spark.sparkContext.accumulator(0)
invalid_count = spark.sparkContext.accumulator(0)

data = [
    ("ORD-001", 99.99),
    ("ORD-002", None),
    ("ORD-003", -5.00),
    ("ORD-004", 150.00),
    ("ORD-005", None),
    ("ORD-006", 0.00),
]
df = spark.createDataFrame(data, ["order_id", "amount"])

# UDF that updates accumulators as a side effect
@udf(returnType=StringType())
def check_and_flag(amount) -> str:
    if amount is None:
        null_count.add(1)
        return "NULL"
    if amount <= 0:
        invalid_count.add(1)
        return "INVALID"
    return "OK"

result = df.withColumn("status", check_and_flag(col("amount")))

# Trigger action — accumulators are updated now
result.show()

print(f"Null amounts:    {null_count.value}")
print(f"Invalid amounts: {invalid_count.value}")

## 4. Built-in Accumulator Types

In [ ]:
# Long accumulator (default)
long_acc = spark.sparkContext.accumulator(0)

# Double accumulator
double_acc = spark.sparkContext.accumulator(0.0)

# Named accumulator — shows in Spark UI under Stages
named_acc = spark.sparkContext.accumulator(0, "processed_rows")

# Increment in a foreach action (reliable — action runs once per row)
df.foreach(lambda row: named_acc.add(1))
print(f"Processed rows: {named_acc.value}")

## 5. Custom Accumulator (AccumulatorParam)

For non-primitive types, subclass `AccumulatorParam`.

In [ ]:
from pyspark import AccumulatorParam

class SetAccumulator(AccumulatorParam):
    """Accumulates unique values into a set."""
    def zero(self, initial_value):
        return set()
    def addInPlace(self, v1, v2):
        return v1 | (v2 if isinstance(v2, set) else {v2})

seen_regions = spark.sparkContext.accumulator(set(), SetAccumulator())

regions_df = spark.createDataFrame([
    ("EU",), ("US",), ("APAC",), ("EU",), ("US",)
], ["region"])

regions_df.foreach(lambda row: seen_regions.add(row["region"]))
print(f"Unique regions seen: {seen_regions.value}")

## 6. Memory Tuning Cheat Sheet

| Config | Default | Tune when... |
|---|---|---|
| `spark.executor.memory` | 1g | Always set explicitly in production |
| `spark.executor.memoryOverhead` | 10% or 384MB | OOM errors from off-heap, native libraries |
| `spark.memory.fraction` | 0.6 | Increase if execution is evicting cache frequently |
| `spark.memory.storageFraction` | 0.5 | Increase if cache is being evicted by execution |
| `spark.sql.shuffle.partitions` | 200 | Decrease for small data, increase for large shuffles |
| `spark.serializer` | Java | Set to Kryo for better performance |

---

## Day 17 Checklist

- [ ] Understand the 3-part memory split: reserved / unified / user
- [ ] Know `spark.memory.fraction` (0.6) and `spark.memory.storageFraction` (0.5)
- [ ] Know that storage and execution memory borrow from each other dynamically
- [ ] Created a named Long accumulator and updated it in a `foreach` action
- [ ] Know that accumulator updates in lazy transformations can be double-counted on retry
- [ ] Know when to increase `memoryOverhead` vs `executor.memory`

**Next:** Day 18 — Pandas API on Spark